# Custom Level 1 Post-Analysis

Copy of `05_compass_level_1_post_analysis`, extended with the same custom `ZONES`/`ZONE_SETS`/`GENOTYPE_SETS` system built in `02b_custom_task_performance_analysis` -- no need for a separate "04b": the HMM fitting in `04_compass_level_1` doesn't depend on maze zones or genotype groupings at all, so its `data_with_states.csv` output is reused as-is.

In [ ]:
# autoreload
%load_ext autoreload
%autoreload 2

# Stashes each saved PDF's title/axis labels in its own metadata (no
# per-plot wiring needed) -- read back by the figure report's Prism cards.
import compass_labyrinth.figure_labels

# Load Project

In [ ]:
from pathlib import Path
import pandas as pd
from compass_labyrinth import load_project


project_path = Path(r'PATH\TO\PROJECT\compass_analysis')

# Import config and metadata
config, cohort_metadata = load_project(project_path=project_path)

### Load the HMM results

In [ ]:
df_hmm = pd.read_csv(project_path / "results" / "compass_level_1" / "data_with_states.csv")
df_hmm

### State Representation: Characteristics

- State 0 --> Stationary (excluded from HMM fitting -- step too small to reliably estimate turn angle, see notebook 04)
- State 1 --> Low Step Length, High Turn Angle --> "Surveillance" state (Red)
- State 2 --> High Step Length, Low Turn Angle --> "Ambulatory" state (Blue)

`MERGE_STATIONARY_INTO_SURVEILLANCE` (set in the "Define Custom Zones and Genotypes" cell below) controls whether state 0 gets folded into state 1 before any plot below runs -- set it to `False` to keep all three states separate.

## Define Custom Zones and Genotypes

Same `ZONES`/`ZONE_SETS`/`GENOTYPE_SETS`/`build_genotype_groups()` system as `02b` -- it's the same 10x10 maze, so the zone cell-number definitions are identical. `build_zone_column()` is new here: it adds a `custom_zone` column to a copy of `df_hmm` by `grid_number` membership (same overlap-safe, order-preserving approach as `02b`'s `build_zone_pivot()`), for use with `compute_state_probability()`'s `column_of_interest` (Plot 2).

Plot 3 (`compute_node_state_medians_over_time`) and Plot 4 (`compute_surveillance_probabilities`) don't need that column -- they already take a `node_type_mapping` dict directly (added alongside this notebook), so pass `node_type_mapping=ZONES` and any two `ZONES` keys as `decision_nodes`/`nondecision_nodes` to compare any custom zone pair, instead of only the fixed `constants.NODE_TYPE_MAPPING` keys.

`MERGE_STATIONARY_INTO_SURVEILLANCE` is a similar on/off toggle for HMM states: `True` (default) merges state 0 (Stationary) into state 1 (Surveillance) via `merge_hmm_states()`, applied to `df_hmm` once at the end of this cell -- every Plot 1-4 cell below reads the already-merged `df_hmm`, so no other cell needs to change when you flip it. Set it to `False` to keep the original 3-state system.

In [ ]:
# Maze zones, zone groupings, genotype groupings, and which genotype groupings
# to generate/offer as a report "Comparison" -- all read from config.yaml now
# (zones/zone_sets/genotype_sets/comparisons), so this notebook, 05b, and the
# figure report share one definition. A different project with a different
# maze layout or genotype labels only needs its own config.yaml, not edits here.
ZONES = config["zones"]
ZONE_SETS = config["zone_sets"]
GENOTYPE_SETS = config["genotype_sets"]
COMPARISONS = config["comparisons"]


def build_genotype_groups(df, set_name):
    """Thin wrapper preserving this notebook's call signature -- see
    compass_labyrinth.utils.build_genotype_groups()."""
    return _build_genotype_groups_pkg(df, set_name, GENOTYPE_SETS)


def build_zone_column(df, zone_names, column_name="custom_zone"):
    """Thin wrapper -- see build_genotype_groups() above."""
    return _build_zone_column_pkg(df, zone_names, ZONES, column_name)


# Whether to merge HMM state 0 (Stationary) into state 1 (Surveillance)
# before any state-based analysis below -- set to False to keep the original
# 3-state system (0=Stationary, 1=Surveillance, 2=Ambulatory) instead.
MERGE_STATIONARY_INTO_SURVEILLANCE = True


def merge_hmm_states(df, merge=MERGE_STATIONARY_INTO_SURVEILLANCE):
    """Return a copy of df with hmm_state relabeled so 0 (Stationary) counts as 1 (Surveillance).

    No-op (besides the copy) if merge=False. Every Plot 1-4 function below
    keys off the literal 'hmm_state' column and a single int/exact-match
    filter, so applying this once (right here, before any plot cell runs)
    makes the merge apply everywhere automatically -- no need to touch each
    plot cell individually.
    """
    df = df.copy()
    if merge:
        df["hmm_state"] = df["hmm_state"].replace(0, 1)
    return df


df_hmm = merge_hmm_states(df_hmm)

import compass_labyrinth.source_data as source_data

_meta = cohort_metadata[["session", "mouseid", "sex", "genotype"]].rename(
    columns={"mouseid": "mouse_id", "genotype": "group_label"}
)
source_data.configure(_meta)

# --- Cache expensive, deterministic computations in the project's own folder ---
# (not this git repo, and specific to this dataset) so repeated notebook runs
# skip recomputation until an actual input -- the dataframe, genotype set,
# zone set, or state/bin settings -- changes. See compass_labyrinth.caching.get_memory().
from compass_labyrinth.caching import get_memory
from compass_labyrinth.utils import (
    build_genotype_groups as _build_genotype_groups_pkg,
    build_zone_column as _build_zone_column_pkg,
)
from compass_labyrinth.post_hoc_analysis import level_1 as _lvl1

memory = get_memory(config)
# Only steps that cost far more to compute than to check are cached: a cache
# hit still has to hash the whole (1.2M-row) input dataframe, ~5 s.
# Patched on the package itself, not just this cell's local name -- later
# cells re-import these via `from ...post_hoc_analysis.level_1 import ...`,
# which would otherwise rebind the plain (uncached) function again.
_lvl1.compute_node_state_medians_over_time = memory.cache(_lvl1.compute_node_state_medians_over_time)
_lvl1.compute_node_velocity_medians_over_time = memory.cache(_lvl1.compute_node_velocity_medians_over_time)


# Plot 1: Heatmap Representations of HMM States

Spatial heatmap of HMM state proportion per grid cell -- Surveillance/Ambulatory (state 1; with `MERGE_STATIONARY_INTO_SURVEILLANCE = True`, Stationary state-0 frames are folded into it, so there is no separate Stationary heatmap) -- with every genotype group overlaid in one figure. Pick a comparison below (one per `comparisons` entry in config.yaml).


In [ ]:
from compass_labyrinth.post_hoc_analysis.level_1 import plot_all_genotype_heatmaps

grid_filename = "Session0001_grid.shp"
# Color scale bounds for every heatmap below -- narrow HEATMAP_VMIN up (e.g.
# 0.7) to spread contrast among values clustered near 1.
HEATMAP_VMIN = 0.3
HEATMAP_VMAX = 1.0

# One set of figures per comparison in config.yaml (genotype_sets /
# comparisons), so the report can offer them as a selectable "Comparison".
# hmm_state=0 (Stationary) is intentionally not plotted here:
# MERGE_STATIONARY_INTO_SURVEILLANCE=True above already folds state 0 into
# state 1 before df_hmm reaches this cell, so a state=0 heatmap would just
# be empty/degenerate as long as that setting stays True.
for genotype_set in COMPARISONS:
    df_geno_all = build_genotype_groups(df_hmm, genotype_set)
    plot_all_genotype_heatmaps(
        config=config,
        show_fig=genotype_set == COMPARISONS[0],  # display the first comparison only; all are saved
        df_hmm=df_geno_all,
        grid_filename=grid_filename,
        highlight_grids=None,
        target_grids="target_zone",
        hmm_state=1,
        cmap='YlOrRd',
        vmin=HEATMAP_VMIN,
        vmax=HEATMAP_VMAX,
        filename_stem="grid_heatmap",
        filename_suffix=f"_{genotype_set}",
        genotype_order=list(GENOTYPE_SETS[genotype_set]),  # config order, e.g. control group first
        figures_dirname="figures/level_1/plot1_heatmap",
    )

print('Target Nodes highlighted in YELLOW')

# Zone-pooled versions (shown in the report under each zone set; "none" = the maps above)
from compass_labyrinth.post_hoc_analysis.level_1 import plot_all_genotype_zone_heatmaps

# Zone sets to plot (names from config.yaml zone_sets) -- add/remove names
ZONE_SET = ["10_rows"]
for zone_set in ZONE_SET:
    for genotype_set in COMPARISONS:
        plot_all_genotype_zone_heatmaps(
            config=config,
            show_fig=(genotype_set, zone_set) == (COMPARISONS[0], ZONE_SET[0]),  # display only the first; all are saved
            df_hmm=build_genotype_groups(df_hmm, genotype_set),
            grid_filename=grid_filename,
            zones=ZONES,
            zone_names=ZONE_SETS[zone_set],
            metric="state",
            hmm_state=1,
            cmap="YlOrRd",
            vmin=HEATMAP_VMIN,
            vmax=HEATMAP_VMAX,
            filename_stem="grid_heatmap",
            filename_suffix=f"_{genotype_set}_{zone_set}",
            genotype_order=list(GENOTYPE_SETS[genotype_set]),
            figures_dirname="figures/level_1/plot1_heatmap",
        )

# Plot 1b: Heatmap Representations of Velocity

Same spatial heatmap mechanism as Plot 1, but colors each grid cell by mean velocity instead of HMM state proportion, for the same three genotype comparisons.


In [ ]:
from compass_labyrinth.post_hoc_analysis.level_1 import compute_mean_velocity, plot_all_genotype_heatmaps

grid_filename = "Session0001_grid.shp"
# Speed color caps per cohort: config.yaml heatmap_vmax (default 300; 400 for allocentric nose)
SPEED_VMAX = config.get("heatmap_vmax") or {}
VELOCITY_VMIN = 0
VELOCITY_VMAX = SPEED_VMAX.get("velocity", 300)

for genotype_set in COMPARISONS:
    df_geno_all = build_genotype_groups(df_hmm, genotype_set)
    plot_all_genotype_heatmaps(
        config=config,
        show_fig=genotype_set == COMPARISONS[0],  # display the first comparison only; all are saved
        df_hmm=df_geno_all,
        grid_filename=grid_filename,
        highlight_grids=None,
        target_grids="target_zone",
        cmap="YlGnBu",
        vmin=VELOCITY_VMIN,
        vmax=VELOCITY_VMAX,
        compute_fn=lambda df, g: compute_mean_velocity(df, g),
        value_col="velocity",
        value_column="mean_velocity",
        metric_label="Mean Velocity",
        filename_stem="velocity_heatmap",
        filename_suffix=f"_{genotype_set}",
        genotype_order=list(GENOTYPE_SETS[genotype_set]),  # config order, e.g. control group first
        figures_dirname="figures/level_1/plot1b_velocity_heatmap",
    )

# Zone-pooled versions (shown in the report under each zone set; "none" = the maps above)
from compass_labyrinth.post_hoc_analysis.level_1 import plot_all_genotype_zone_heatmaps

# Zone sets to plot (names from config.yaml zone_sets) -- add/remove names
ZONE_SET = ["10_rows"]
for zone_set in ZONE_SET:
    for genotype_set in COMPARISONS:
        plot_all_genotype_zone_heatmaps(
            config=config,
            show_fig=(genotype_set, zone_set) == (COMPARISONS[0], ZONE_SET[0]),  # display only the first; all are saved
            df_hmm=build_genotype_groups(df_hmm, genotype_set),
            grid_filename=grid_filename,
            zones=ZONES,
            zone_names=ZONE_SETS[zone_set],
            metric="velocity",
            cmap="YlGnBu",
            vmin=VELOCITY_VMIN,
            vmax=VELOCITY_VMAX,
            metric_label="Mean Velocity",
            filename_stem="velocity_heatmap",
            filename_suffix=f"_{genotype_set}_{zone_set}",
            genotype_order=list(GENOTYPE_SETS[genotype_set]),
            figures_dirname="figures/level_1/plot1b_velocity_heatmap",
        )

# Plots 1c-1f: Sternum Speed, Nose Speed and Head Angle Heatmaps

Same mechanism as Plot 1b, for other body points (cells are still assigned by the belly, the point the whole pipeline uses). Nose, sternum and tail base come from the DLC output the pipeline was built from (`dlc_results/SessionXXXX_withGrids.csv`), joined by `s_no`.

- **1c, sternum speed**: how fast the sternum moves relative to the maze, on the same color scale as 1b (belly) for comparison.
- **1d, allocentric nose speed**: how fast the nose moves relative to the maze.
- **1e, egocentric nose speed**: how fast the nose moves relative to the body -- its position relative to the sternum, rotated so the forward axis points from the tail base to the sternum -- i.e. head movement with the body's own movement and turning removed.
- **1f, absolute head angle**: how far the nose points away from the body axis (tail base -> sternum), either side; 0 = straight ahead. (Which side, split by direction of travel: Plot 7.)

Same rules as the pipeline's velocity: a point is used only on frames with likelihood > 0.6 (others are dropped for that measure, not interpolated), and speed = distance between consecutive kept rows x 5 fps. Every frame's values are saved to `results/compass_level_1/data_with_kinematics.csv`.

In [ ]:
from compass_labyrinth.post_hoc_analysis.level_1 import add_nose_speeds, compute_mean_metric, plot_all_genotype_heatmaps

grid_filename = "Session0001_grid.shp"

# Nose / sternum / tail base for every analyzed frame: speeds and head angle
df_kin = add_nose_speeds(df_hmm, config, likelihood_threshold=0.6, fps=5.0)
df_kin.to_csv(project_path / "results" / "compass_level_1" / "data_with_kinematics.csv", index=False)

BODY_PLOTS = [
    # (column, figure folder, filename stem, label, colormap, scale min, scale max, body part in the title)
    ("sternum_speed", "plot1c_sternum_speed", "sternum_speed", "Sternum Speed", "YlGnBu", 0, SPEED_VMAX.get("sternum_speed", 300), "Sternum"),
    ("nose_speed_allo", "plot1d_nose_speed_allocentric", "nose_speed_allocentric", "Allocentric Nose Speed", "YlGnBu", 0, SPEED_VMAX.get("nose_speed_allo", 400), "Nose"),
    ("nose_speed_ego", "plot1e_nose_speed_egocentric", "nose_speed_egocentric", "Egocentric Nose Speed", "YlGnBu", 0, SPEED_VMAX.get("nose_speed_ego", 300), "Nose"),
    ("head_angle_abs", "plot1f_head_angle_absolute", "head_angle_absolute", "Absolute Head Angle (deg)", "magma_r", 0, 60, "Nose"),
]
for column, folder, stem, label, cmap, vmin, vmax, body_part in BODY_PLOTS:
    for genotype_set in COMPARISONS:
        plot_all_genotype_heatmaps(
            config=config,
            show_fig=genotype_set == COMPARISONS[0],  # display the first comparison only; all are saved
            df_hmm=build_genotype_groups(df_kin, genotype_set),
            grid_filename=grid_filename,
            highlight_grids=None,
            target_grids="target_zone",
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
            compute_fn=lambda df, g, column=column: compute_mean_metric(df, g, column),
            value_col=column,
            value_column=column,
            metric_label=label,
            filename_stem=stem,
            filename_suffix=f"_{genotype_set}",
            genotype_order=list(GENOTYPE_SETS[genotype_set]),  # config order, e.g. control group first
            tracking_label=body_part,
            figures_dirname=f"figures/level_1/{folder}",
        )

# Plot 2: Probability of Surveillance across Custom Zones

This workflow computes and visualizes the probability of HMM state occupancy across custom zones (instead of the fixed `node_type`/`region` categories).

### Recommended Use:
1. Pick a `ZONE_SET` from `ZONE_SETS` and a `GENOTYPE_SET` from `GENOTYPE_SETS`.
2. `build_zone_column()` adds a `custom_zone` column to `df_hmm` by grid_number.
3. Use `compute_state_probability()` (with `column_of_interest="custom_zone"`) to get HMM state proportions per zone.
4. Use `plot_state_probability_boxplot()` to visualize across genotypes.

In [ ]:
from compass_labyrinth.post_hoc_analysis.level_1 import (
    compute_state_probability,
    plot_state_probability_boxplot,
)

# Zone sets to plot (names from config.yaml zone_sets) -- add/remove names; one figure per comparison x set
ZONE_SET = ["9_corners", "six_zones", "5_rows"]
STATE = 1  # Slow / Surveillance

# Step 1: per-session state proportion per zone -- once per zone set, on the
# raw genotypes; relabeled per comparison. Keyed (comparison, zone set).
state_count_by_comparison = {}
for zone_set in ZONE_SET:
    ZONE_NAMES = ZONE_SETS[zone_set]
    state_count_raw = compute_state_probability(
        df_hmm=build_zone_column(df_hmm, ZONE_NAMES),
        column_of_interest="custom_zone",
        values_displayed=ZONE_NAMES,
        state=STATE,
    )
    for gs in COMPARISONS:
        state_count_by_comparison[(gs, zone_set)] = build_genotype_groups(state_count_raw, gs)

# Step 2: one boxplot per comparison x zone set
FIRST = (COMPARISONS[0], ZONE_SET[0])
for (GENOTYPE_SET, zone_set), state_count_df in state_count_by_comparison.items():
    plot_state_probability_boxplot(
        config=config,
        show_fig=(GENOTYPE_SET, zone_set) == FIRST,  # display only the first comparison x zone set; all are saved
        state_count_df=state_count_df,
        column_of_interest="custom_zone",
        state=STATE,
        filename_suffix=f"_{GENOTYPE_SET}_{zone_set}",
        figures_dirname="figures/level_1/plot2_state_probability",
    )

In [ ]:
from compass_labyrinth.post_hoc_analysis.level_1 import run_pairwise_ttests


# T-tests amongst genotype groups per zone, for each comparison x zone set
ttest_results_by_comparison = {}
for (GENOTYPE_SET, zone_set), state_count_df in state_count_by_comparison.items():
    ttest_results = run_pairwise_ttests(state_count_df, column_of_interest="custom_zone")
    ttest_results_by_comparison[(GENOTYPE_SET, zone_set)] = ttest_results
    print(f"\n######## {GENOTYPE_SET} | {zone_set} ########")
    print(ttest_results.sort_values("p_value"))

In [ ]:
from pathlib import Path

# Save pairwise t-tests alongside the other level-1 results, for the figure
# report -- tagged _<comparison>_<zone set> so each shows under its selection
stats_dir = Path(config["project_path_full"]) / "results" / "level_1" / "plot2_state_probability"
stats_dir.mkdir(parents=True, exist_ok=True)

for (GENOTYPE_SET, zone_set), ttest_results in ttest_results_by_comparison.items():
    ttest_results.to_csv(stats_dir / f"pairwise_ttests_{GENOTYPE_SET}_{zone_set}.csv", index=False)

# Plot 2b: Mean Velocity across Custom Zones

Same mechanism as Plot 2, but computes mean velocity per zone (`compute_zone_velocity()`) instead of HMM state proportion -- `plot_state_probability_boxplot()`'s `value_col`/`metric_label` parameters let it plot that instead of a state probability.

### Recommended Use:
1. Pick a `ZONE_SET` from `ZONE_SETS` and a `GENOTYPE_SET` from `GENOTYPE_SETS`.
2. `build_zone_column()` adds a `custom_zone` column to `df_hmm` by grid_number.
3. Use `compute_zone_velocity()` to get mean velocity per zone, per session.
4. Use `plot_state_probability_boxplot()` (with `value_col="velocity"`, `metric_label="Mean Velocity"`) to visualize across genotypes.

In [ ]:
from compass_labyrinth.post_hoc_analysis.level_1 import (
    compute_zone_velocity,
    plot_state_probability_boxplot,
)

# Zone sets to plot (names from config.yaml zone_sets) -- add/remove names; one figure per comparison x set
ZONE_SET = ["5_columns", "six_zones", "5_rows"]

# Step 1: per-session mean velocity per zone -- once per zone set, on the raw
# genotypes; relabeled per comparison. Keyed (comparison, zone set).
velocity_zone_by_comparison = {}
for zone_set in ZONE_SET:
    ZONE_NAMES = ZONE_SETS[zone_set]
    velocity_zone_raw = compute_zone_velocity(
        df_hmm=build_zone_column(df_hmm, ZONE_NAMES),
        column_of_interest="custom_zone",
        values_displayed=ZONE_NAMES,
    )
    for gs in COMPARISONS:
        velocity_zone_by_comparison[(gs, zone_set)] = build_genotype_groups(velocity_zone_raw, gs)

# Step 2: one boxplot per comparison x zone set
FIRST = (COMPARISONS[0], ZONE_SET[0])
for (GENOTYPE_SET, zone_set), velocity_zone_df in velocity_zone_by_comparison.items():
    plot_state_probability_boxplot(
        config=config,
        show_fig=(GENOTYPE_SET, zone_set) == FIRST,  # display only the first comparison x zone set; all are saved
        state_count_df=velocity_zone_df,
        column_of_interest="custom_zone",
        value_col="velocity",
        metric_label="Mean Velocity",
        filename_stem="mean_velocity",
        palette="Set1",
        filename_suffix=f"_{GENOTYPE_SET}_{zone_set}",
        figures_dirname="figures/level_1/plot2b_velocity",
    )

In [ ]:
from compass_labyrinth.post_hoc_analysis.level_1 import run_pairwise_ttests


# T-tests amongst genotype groups per zone, for each comparison x zone set
ttest_results_velocity_by_comparison = {}
for (GENOTYPE_SET, zone_set), velocity_zone_df in velocity_zone_by_comparison.items():
    ttest_results_velocity = run_pairwise_ttests(velocity_zone_df, column_of_interest="custom_zone", value_col="velocity")
    ttest_results_velocity_by_comparison[(GENOTYPE_SET, zone_set)] = ttest_results_velocity
    print(f"\n######## {GENOTYPE_SET} | {zone_set} ########")
    print(ttest_results_velocity.sort_values("p_value"))

In [ ]:
from pathlib import Path

# Save pairwise t-tests alongside the other level-1 results, for the figure
# report -- tagged _<comparison>_<zone set> so each shows under its selection
stats_dir = Path(config["project_path_full"]) / "results" / "level_1" / "plot2b_velocity"
stats_dir.mkdir(parents=True, exist_ok=True)

for (GENOTYPE_SET, zone_set), ttest_results_velocity in ttest_results_velocity_by_comparison.items():
    ttest_results_velocity.to_csv(
        stats_dir / f"pairwise_ttests_velocity_{GENOTYPE_SET}_{zone_set}.csv", index=False
    )

# Plot 3: Probability of States over Time

This workflow analyzes how the probability of being in a specific HMM state evolves over time across a set of custom zones. It computes median probabilities across sessions using sliding time bins and plots one curve per zone.

### Recommended Use:
1. Pick a `GENOTYPE_SET` and a `ZONE_SET` from `ZONE_SETS` (any number of zones -- not limited to two).
2. Set a time window (`lower_limit` to `upper_limit`) and a `bin_size` to compute time bins.
3. Use `compute_node_state_medians_over_time()` with `zone_names=ZONE_NAMES` and `node_type_mapping=ZONES` to calculate median state occupancy at each zone.
4. Visualize the trajectory of state occupancy using `plot_node_state_median_curve()`.

In [ ]:
from compass_labyrinth.post_hoc_analysis.level_1 import (
    get_max_session_row_bracket,
    get_min_session_row_bracket,
    compute_node_state_medians_over_time,
    build_genotype_zone_palette,
    plot_node_state_median_curve,
)

# Zone sets to plot (names from config.yaml zone_sets) -- add/remove names; one figure per comparison x set
ZONE_SET = ["corners_cols_reward", "3_rows"]

lower_limit = 0
upper_limit = get_max_session_row_bracket(df_hmm)
threshold = get_min_session_row_bracket(df_hmm)  # Only show bins where all sessions are present
bin_size = 2000
figure_ylimit = (0.3, 1.1)


def relabel_medians(deci_df, set_name):
    """A raw-genotype temporal-median table relabeled for one comparison
    (genotype -> its group), with the genotype/zone legend label rebuilt."""
    out = build_genotype_groups(deci_df, set_name)
    out["genotype_node_type"] = out["genotype"] + " , " + out["node_type"]
    return out


FIRST = (COMPARISONS[0], ZONE_SET[0])
for zone_set in ZONE_SET:
    # Per-session medians over time -- the expensive step (cached), computed
    # once per zone set on the raw genotypes; each comparison relabels it.
    # node_type_mapping=ZONES looks the zones up in the notebook's own ZONES.
    deci_df_raw = compute_node_state_medians_over_time(
        df_hmm=df_hmm,
        state_types=[1],
        lower_lim=lower_limit,
        upper_lim=upper_limit,
        bin_size=bin_size,
        zone_names=ZONE_SETS[zone_set],
        node_type_mapping=ZONES,
    )
    # Optional filter to only plot early session bins
    deci_df_raw = deci_df_raw.loc[deci_df_raw.time_bins < threshold]

    for GENOTYPE_SET in COMPARISONS:
        deci_df = relabel_medians(deci_df_raw, GENOTYPE_SET)
        # One color per genotype; zones get their own marker + line style
        palette = build_genotype_zone_palette(deci_df)

        # Time-evolving median probability curves
        plot_node_state_median_curve(
            config=config,
            show_fig=(GENOTYPE_SET, zone_set) == FIRST,  # display only the first comparison x zone set; all are saved
            deci_df=deci_df,
            palette=palette,
            figure_ylimit=figure_ylimit,
            metric_label="Median Probability of Surveillance",
            fig_title=f'Median Probability of Surveillance State across {zone_set} zones',
            filename_suffix=f"_{GENOTYPE_SET}_{zone_set}",
            figures_dirname="figures/level_1/plot3_temporal_median",
        )

# Plot 3b: Median Velocity over Time across Custom Zones

Same mechanism as Plot 3, but computes median per-grid-cell velocity per zone (`compute_node_velocity_medians_over_time()`) instead of HMM state proportion -- `plot_node_state_median_curve()`'s `value_col`/`metric_label` parameters let it plot that instead.

### Recommended Use:
1. Pick a `GENOTYPE_SET` and a `ZONE_SET` from `ZONE_SETS` (any number of zones -- not limited to two).
2. Set a time window (`lower_limit` to `upper_limit`) and a `bin_size` to compute time bins.
3. Use `compute_node_velocity_medians_over_time()` with `zone_names=ZONE_NAMES` and `node_type_mapping=ZONES` to calculate median velocity at each zone.
4. Visualize the trajectory of velocity using `plot_node_state_median_curve()` (with `value_col="median_velocity"`, `metric_label="Median Velocity"`).

In [ ]:
from compass_labyrinth.post_hoc_analysis.level_1 import (
    get_max_session_row_bracket,
    get_min_session_row_bracket,
    compute_node_velocity_medians_over_time,
    build_genotype_zone_palette,
    plot_node_state_median_curve,
)

# Zone sets to plot (names from config.yaml zone_sets) -- add/remove names; one figure per comparison x set
ZONE_SET = ["corners_cols_reward", "3_rows"]

lower_limit = 0
upper_limit = get_max_session_row_bracket(df_hmm)
threshold = get_min_session_row_bracket(df_hmm)  # Only show bins where all sessions are present
bin_size = 2000
# Real per-cell mean velocity in this dataset ranges ~22-320 (see Plot 1b),
# not [0, 1] like Plot 3's state-probability figure_ylimit (the plot widens
# it if the data needs more).
figure_ylimit = (0, 300)

FIRST = (COMPARISONS[0], ZONE_SET[0])
for zone_set in ZONE_SET:
    # Per-session medians over time -- the expensive step (cached), computed
    # once per zone set on the raw genotypes; each comparison relabels it
    # (relabel_medians() is defined in the Plot 3 cell above).
    velocity_deci_df_raw = compute_node_velocity_medians_over_time(
        df_hmm=df_hmm,
        lower_lim=lower_limit,
        upper_lim=upper_limit,
        bin_size=bin_size,
        zone_names=ZONE_SETS[zone_set],
        node_type_mapping=ZONES,
    )
    # Optional filter to only plot early session bins
    velocity_deci_df_raw = velocity_deci_df_raw.loc[velocity_deci_df_raw.time_bins < threshold]

    for GENOTYPE_SET in COMPARISONS:
        velocity_deci_df = relabel_medians(velocity_deci_df_raw, GENOTYPE_SET)
        velocity_palette = build_genotype_zone_palette(velocity_deci_df)

        # Time-evolving median velocity curves
        plot_node_state_median_curve(
            config=config,
            show_fig=(GENOTYPE_SET, zone_set) == FIRST,  # display only the first comparison x zone set; all are saved
            deci_df=velocity_deci_df,
            palette=velocity_palette,
            figure_ylimit=figure_ylimit,
            value_col="median_velocity",
            metric_label="Median Velocity",
            filename_stem="temporal_median_velocity_curve",
            fig_title=f'Median Velocity across {zone_set} zones',
            filename_suffix=f"_{GENOTYPE_SET}_{zone_set}",
            figures_dirname="figures/level_1/plot3b_temporal_median_velocity",
        )

# Plot 4: Surveillance Probability by Bout Type

This workflow evaluates behavioral surveillance patterns at a custom zone across navigational bouts in the maze. It segments the session into bouts, computes surveillance probability for each bout, and visualizes the average surveillance behavior across successful and unsuccessful bouts.

### Recommended Use:
1. Pick a `GENOTYPE_SET` and a `ZONES` key (`SURVEILLANCE_ZONE`) to compute surveillance probability at.
2. Use `assign_bout_indices_from_entry_zone()` to segment sessions into bouts.
3. Run `compute_surveillance_probabilities()` (with `node_type_mapping=ZONES`) to calculate surveillance at that zone for each bout.
4. Use `plot_surveillance_by_bout()` to generate a grouped barplot comparing surveillance probability between successful and unsuccessful bouts.

In [ ]:
from compass_labyrinth.behavior.behavior_metrics.task_performance_analysis import assign_bout_indices_from_entry_zone
from compass_labyrinth.post_hoc_analysis.level_1 import (
    compute_surveillance_probabilities,
    plot_surveillance_by_bout,
)
from compass_labyrinth.constants import ENTRY_ZONE


# Zones to measure surveillance at (names from config.yaml zones) -- add/remove
# names; one figure per comparison x zone
SURVEILLANCE_ZONE = ["top_corners"]

# Assign Bout Numbers -- once, on the raw genotypes (bouts are per session)
# Bout = Entry zone --> Other non-entry nodes --> Entry zone
# entry_zone is [0, 1] -- delimiter_nodes=ENTRY_ZONE catches sessions that
# never happen to have tracked data in cell 0 exactly but do in its neighbor
# cell 1 (see task_performance_analysis for the original fix).
df_hmm_bouts = assign_bout_indices_from_entry_zone(
    df_hmm,
    delimiter_nodes=ENTRY_ZONE,
)

# Surveillance probability at each zone by Bout type, once per zone on the raw
# genotypes, relabeled per comparison. Keyed (comparison, zone).
# Successful -> reached the target at least once / Unsuccessful -> didn't
surveillance_by_comparison = {}
for zone_set in SURVEILLANCE_ZONE:
    index_df_raw, median_df_raw = compute_surveillance_probabilities(
        df_hmm=df_hmm_bouts,
        decision_nodes=zone_set,
        node_type_mapping=ZONES,
    )
    for gs in COMPARISONS:
        surveillance_by_comparison[(gs, zone_set)] = (
            build_genotype_groups(index_df_raw, gs), build_genotype_groups(median_df_raw, gs)
        )

# Barplot to depict the above with ttest-ind pvalue, one per comparison x zone
FIRST = (COMPARISONS[0], SURVEILLANCE_ZONE[0])
for (GENOTYPE_SET, zone_set), (index_df, median_df) in surveillance_by_comparison.items():
    plot_surveillance_by_bout(
        config=config,
        show_fig=(GENOTYPE_SET, zone_set) == FIRST,  # display only the first comparison x zone set; all are saved
        median_df=median_df,
        ylim=1,
        filename_suffix=f"_{GENOTYPE_SET}_{zone_set}",
        figures_dirname="figures/level_1/plot4_surveillance_bout",
    )

In [ ]:
from compass_labyrinth.post_hoc_analysis.level_1 import run_within_genotype_mixedlm_with_fdr


# LMM for same genotype group comparison across Bout types, for each comparison x zone
within_by_comparison = {}
for (GENOTYPE_SET, zone_set), (index_df, median_df) in surveillance_by_comparison.items():
    df_within = run_within_genotype_mixedlm_with_fdr(index_df)
    within_by_comparison[(GENOTYPE_SET, zone_set)] = df_within
    print(f"Within-Genotype (Successful vs Unsuccessful), {GENOTYPE_SET} | {zone_set}:")
    print(df_within)

In [ ]:
from compass_labyrinth.post_hoc_analysis.level_1 import test_across_genotypes_per_bout


# T-tests across genotype groups under Successful / Unsuccessful Bouts, for each comparison x zone
across_by_comparison = {}
for (GENOTYPE_SET, zone_set), (index_df, median_df) in surveillance_by_comparison.items():
    df_across_success = test_across_genotypes_per_bout(median_df, bout_type='Successful')
    df_across_unsuccess = test_across_genotypes_per_bout(median_df, bout_type='Unsuccessful')
    across_by_comparison[(GENOTYPE_SET, zone_set)] = (df_across_success, df_across_unsuccess)

    print(f"\n######## {GENOTYPE_SET} | {zone_set} ########")
    print("\n Across Genotypes (Successful only):")
    print(df_across_success)
    print("\n Across Genotypes (Unsuccessful only):")
    print(df_across_unsuccess)

In [ ]:
from pathlib import Path

# Save surveillance-by-bout stats alongside the other level-1 results, for the
# figure report -- tagged _<comparison>_<zone> so each shows under its selection
stats_dir = Path(config["project_path_full"]) / "results" / "level_1" / "plot4_surveillance_bout"
stats_dir.mkdir(parents=True, exist_ok=True)

for (GENOTYPE_SET, zone_set), df_within in within_by_comparison.items():
    suffix = f"_{GENOTYPE_SET}_{zone_set}"
    df_within.to_csv(stats_dir / f"within_genotype_mixedlm{suffix}.csv", index=False)
    df_across_success, df_across_unsuccess = across_by_comparison[(GENOTYPE_SET, zone_set)]
    df_across_success.to_csv(stats_dir / f"across_genotypes_successful{suffix}.csv", index=False)
    df_across_unsuccess.to_csv(stats_dir / f"across_genotypes_unsuccessful{suffix}.csv", index=False)

# Plot 4b: Surveillance by Bout Type over Time

Each bout in the time bin of its first frame (`pct_bins`: 5 equal parts of the session; `frame_bins`: frames 0-10k ... 40k-50k). Successful | Unsuccessful panels; per mouse, the median over its bouts in that bin.

In [ ]:
from compass_labyrinth.post_hoc_analysis.level_1 import (
    surveillance_by_bout_bins,
    plot_surveillance_by_bout_bins,
    bout_bins_mixedlm,
    bout_bins_ttests,
)

stats_dir_4b = Path(config["project_path_full"]) / "results" / "level_1" / "plot4b_surveillance_bout_time_bins"
stats_dir_4b.mkdir(parents=True, exist_ok=True)

for zone_set in SURVEILLANCE_ZONE:
    for binning in ("pct_bins", "frame_bins"):
        _, median_bins_raw = surveillance_by_bout_bins(df_hmm_bouts, zone_set, binning, node_type_mapping=ZONES)
        for gs in COMPARISONS:
            median_bins = build_genotype_groups(median_bins_raw, gs)
            suffix = f"_{gs}_{zone_set}"
            plot_surveillance_by_bout_bins(
                config=config,
                show_fig=(gs, zone_set, binning) == (COMPARISONS[0], SURVEILLANCE_ZONE[0], "pct_bins"),
                median_df=median_bins,
                binning=binning,
                filename_suffix=suffix,
                figures_dirname="figures/level_1/plot4b_surveillance_bout_time_bins",
            )
            bout_bins_mixedlm(median_bins).to_csv(stats_dir_4b / f"time_bins_mixedlm{suffix}_{binning}.csv", index=False)
            bout_bins_ttests(median_bins, binning).to_csv(
                stats_dir_4b / f"time_bins_across_genotypes{suffix}_{binning}.csv", index=False)

# Plot 5: Bout Depth and Direction of Travel

The maze is one path with no branches, so every cell has a **position along the path** (`FORWARD_PATH`: entry 0 -> lane 0 -> U-turn -> lane 1 back -> ... -> reward after lane 7 -> overshoot lanes 8-9 -> terminal). Each frame gets:

- **direction**: sign of the net path change from 10 frames before to 10 frames after (+-2 s at 5 fps; jumps of > 3 positions ignored as tracking errors) -- forward (toward the reward/terminal) or backward; a pause (no net movement) takes the direction of the nearer moving frame in time (it splits at its midpoint), so it counts toward the stretches it interrupts
- **phase**, relative to the bout's turning point (a `moving` column marks which frames were pauses):
  - bouts that **reach the reward** turn at their **first arrival** there: before it, forward = **outbound**, backward = **backtrack**; after it, by zone -- reward zone = **at reward**; overshoot: forward = **overshoot forward**, backward = **overshoot backward**; path: backward = **return**, forward = **re-entry** (7 phases, every frame)
  - other bouts turn at their **furthest point** (the stay there, split at its midpoint: first half outbound, second half return): before it, forward = **outbound**, backward = **backtrack**; after it, backward = **return**, forward = **re-entry**

The figure shows how deep bouts go (% of each session's bouts turning back in each lane) and how often mice move against their bout's phase. Every frame's label is saved to `results/compass_level_1/data_with_phases.csv` (same rows as `data_with_states.csv`).

In [ ]:
from compass_labyrinth.behavior.behavior_metrics.task_performance_analysis import assign_bout_indices_from_entry_zone
from compass_labyrinth.post_hoc_analysis.level_1 import (
    label_bout_directions,
    frame_phases,
    summarize_bouts,
    bout_depth_by_session,
    phase_by_session,
    depth_table,
    phase_table,
    plot_bout_depth,
)
from compass_labyrinth.constants import ENTRY_ZONE

DIRECTION_WINDOW = 10  # frames before/after for direction (10 = +-2 s at 5 fps)

# Per frame: path position, direction, bout phase -- once, on the raw genotypes
df_hmm_bouts = assign_bout_indices_from_entry_zone(df_hmm, delimiter_nodes=ENTRY_ZONE)
labeled_frames = label_bout_directions(df_hmm_bouts, window=DIRECTION_WINDOW)
bouts_raw = summarize_bouts(labeled_frames)  # one row per complete bout

# Every frame with its category, next to the HMM results
frame_phases(df_hmm_bouts, labeled_frames).to_csv(
    project_path / "results" / "compass_level_1" / "data_with_phases.csv", index=False
)

bout_direction_by_comparison = {}
for GENOTYPE_SET in COMPARISONS:
    bouts = build_genotype_groups(bouts_raw, GENOTYPE_SET)
    depth_df, phase_df = bout_depth_by_session(bouts), phase_by_session(bouts)
    bout_direction_by_comparison[GENOTYPE_SET] = (depth_df, phase_df)
    plot_bout_depth(
        config=config,
        show_fig=GENOTYPE_SET == COMPARISONS[0],  # display the first comparison only; all are saved
        depth_df=depth_df,
        phase_df=phase_df,
        filename_suffix=f"_{GENOTYPE_SET}",
        figures_dirname="figures/level_1/plot5_bout_depth",
    )

depth_df, phase_df = bout_direction_by_comparison[COMPARISONS[0]]
display(depth_table(depth_df), phase_table(phase_df))

In [ ]:
from pathlib import Path

# Save the depth and phase tables (mean (SE) across sessions) for the figure
# report -- tagged _<comparison> so each shows under its selection
stats_dir = Path(config["project_path_full"]) / "results" / "level_1" / "plot5_bout_depth"
stats_dir.mkdir(parents=True, exist_ok=True)

for GENOTYPE_SET, (depth_df, phase_df) in bout_direction_by_comparison.items():
    depth_table(depth_df).to_csv(stats_dir / f"bout_depth_{GENOTYPE_SET}.csv", index=False)
    phase_table(phase_df).to_csv(stats_dir / f"bout_phase_{GENOTYPE_SET}.csv", index=False)

# Plot 5b: Time by Direction, Reward-Reaching vs. Other Bouts

From Plot 5's per-frame phases, split by whether the bout reached the reward: for each genotype group, the fraction of those bouts' frames in each of Plot 5's phases (reward-reaching bouts: 7, incl. **at reward** and **overshoot forward / backward**; others: 4) -- the fractions sum to 1 per session (mean +- SE across sessions). Splitting by bout type keeps short aborted bouts from diluting the deep ones.

In [ ]:
from compass_labyrinth.post_hoc_analysis.level_1 import (
    BOUT_TYPES,
    phase_fractions_by_session,
    phase_fraction_ttests,
    plot_phase_fractions,
)

# Reuses Plot 5's bouts_raw (one row per complete bout, raw genotypes)
phase_fraction_by_comparison = {}
for GENOTYPE_SET in COMPARISONS:
    frac_df = phase_fractions_by_session(build_genotype_groups(bouts_raw, GENOTYPE_SET))
    phase_fraction_by_comparison[GENOTYPE_SET] = frac_df
    for bout_type in BOUT_TYPES.values():
        plot_phase_fractions(
            config=config,
            show_fig=GENOTYPE_SET == COMPARISONS[0],  # display the first comparison only; all are saved
            frac_df=frac_df,
            bout_type=bout_type,
            filename_suffix=f"_{GENOTYPE_SET}",
            figures_dirname="figures/level_1/plot5b_phase_fractions",
        )

In [ ]:
from pathlib import Path

# Welch t-tests between genotype groups per phase and bout type (FDR across
# the 4 phases), tagged _<comparison> for the figure report
stats_dir = Path(config["project_path_full"]) / "results" / "level_1" / "plot5b_phase_fractions"
stats_dir.mkdir(parents=True, exist_ok=True)

for GENOTYPE_SET, frac_df in phase_fraction_by_comparison.items():
    ttests = phase_fraction_ttests(frac_df)
    ttests.to_csv(stats_dir / f"phase_fraction_ttests_{GENOTYPE_SET}.csv", index=False)
    print(f"\n######## {GENOTYPE_SET} ########")
    print(ttests.sort_values("p_value"))

# Plot 5c / 5d: Bout Time and Bout Distance

Analyzed frames per complete bout (entry zone -> maze -> entry zone), Successful (reached the target zone, as Plot 4) vs Unsuccessful, plus the 8 parts of a successful bout, which add up to all its frames: Plot 5's phases (outbound, backtrack, overshoot forward / backward, return, re-entry) and the reward zone split into the first visit and later visits. Per mouse, the median over its bouts; log axes. Also by time bin of the bout's first frame (`pct_bins`, `frame_bins`). Tests on log10 lengths.

**5d** repeats every figure and test with distance instead of time: cells moved along the path (frame to frame; jumps of more than 3 positions are tracking errors and count 0, as in Plot 5).


In [ ]:
from compass_labyrinth.post_hoc_analysis.level_1 import (
    BOUT_MEASURES,
    bout_lengths,
    bout_length_medians,
    plot_bout_length,
    plot_bout_length_bins,
    bout_length_ttests,
    bout_length_mixedlm,
    bout_bins_mixedlm,
    bout_bins_ttests,
)

# One row per complete bout, raw genotypes (Plot 5's df_hmm_bouts and phases)
bout_len_raw = bout_lengths(df_hmm_bouts, labeled_frames)

# 5c = bout time (frames), 5d = bout distance (cells moved along the path): same figures and tests
BOUT_STEPS = {"frames": "plot5c_bout_length", "cells": "plot5d_bout_distance"}
for measure, step in BOUT_STEPS.items():
    figures_dir = f"figures/level_1/{step}"
    stats_dir = Path(config["project_path_full"]) / "results" / "level_1" / step
    stats_dir.mkdir(parents=True, exist_ok=True)
    stem = BOUT_MEASURES[measure]["stem"]

    for GENOTYPE_SET in COMPARISONS:
        bout_len = build_genotype_groups(bout_len_raw, GENOTYPE_SET)
        medians = bout_length_medians(bout_len, measure=measure)
        plot_bout_length(
            config=config,
            show_fig=(GENOTYPE_SET, measure) == (COMPARISONS[0], "frames"),  # display only the first; all are saved
            bouts=bout_len,
            median_df=medians,
            measure=measure,
            filename_suffix=f"_{GENOTYPE_SET}",
            figures_dirname=figures_dir,
        )
        bout_length_ttests(medians, measure).to_csv(stats_dir / f"{stem}_ttests_{GENOTYPE_SET}.csv", index=False)
        bout_length_mixedlm(bout_len, measure).to_csv(stats_dir / f"{stem}_mixedlm_{GENOTYPE_SET}.csv", index=False)

        for binning in ("pct_bins", "frame_bins"):
            medians_bins = bout_length_medians(bout_len, binning, measure)
            plot_bout_length_bins(
                config=config,
                show_fig=False,
                median_df=medians_bins,
                binning=binning,
                measure=measure,
                filename_suffix=f"_{GENOTYPE_SET}",
                figures_dirname=figures_dir,
            )
            suffix = f"_{GENOTYPE_SET}_{binning}"
            bout_bins_mixedlm(medians_bins, value=f"log10_{measure}").to_csv(
                stats_dir / f"time_bins_mixedlm{suffix}.csv", index=False)
            bout_bins_ttests(medians_bins, binning, value=f"log10_{measure}").to_csv(
                stats_dir / f"time_bins_across_genotypes{suffix}.csv", index=False)


# Plot 5e: How Far Backtracks and Re-entries Go

Each run of frames moving the same way that starts as a backtrack or re-entry (Plot 5's phases) is one occurrence -- a re-entry carries on into the reward and overshoot until the mouse turns; only occurrences that move at least 1 cell count. Per mouse: occurrences per bout, % of occurrences moving 1 / 2-4 / 5-9 / 10+ cells, and the lane where it turned around. All complete bouts, successful or not. Also by time bin of the bout's first frame (`pct_bins`, `frame_bins`): occurrences per bout, mean cells moved, mean turnaround lane.

In [ ]:
from compass_labyrinth.post_hoc_analysis.level_1 import (
    reversal_occurrences,
    reversals_by_session,
    plot_reversal_distance,
    reversal_distance_ttests,
    reversals_by_bins,
    plot_reversal_bins,
    reversal_bins_tests,
)

# Reuses Plot 5's labeled_frames and bouts_raw (raw genotypes)
occurrences_raw = reversal_occurrences(labeled_frames)

stats_dir_5d = Path(config["project_path_full"]) / "results" / "level_1" / "plot5e_reversal_distance"
stats_dir_5d.mkdir(parents=True, exist_ok=True)

for GENOTYPE_SET in COMPARISONS:
    per_session, distances, lanes = reversals_by_session(
        build_genotype_groups(occurrences_raw, GENOTYPE_SET), build_genotype_groups(bouts_raw, GENOTYPE_SET)
    )
    plot_reversal_distance(
        config=config,
        show_fig=GENOTYPE_SET == COMPARISONS[0],  # display the first comparison only; all are saved
        per_session=per_session,
        distances=distances,
        lanes=lanes,
        filename_suffix=f"_{GENOTYPE_SET}",
        figures_dirname="figures/level_1/plot5e_reversal_distance",
    )
    reversal_distance_ttests(per_session, distances, lanes).to_csv(
        stats_dir_5d / f"reversal_distance_ttests_{GENOTYPE_SET}.csv", index=False)

    for binning in ("pct_bins", "frame_bins"):
        binned = reversals_by_bins(
            build_genotype_groups(occurrences_raw, GENOTYPE_SET), build_genotype_groups(bouts_raw, GENOTYPE_SET),
            df_hmm_bouts, binning,
        )
        plot_reversal_bins(
            config=config,
            show_fig=False,
            binned=binned,
            binning=binning,
            filename_suffix=f"_{GENOTYPE_SET}",
            figures_dirname="figures/level_1/plot5e_reversal_distance",
        )
        lmm, tt = reversal_bins_tests(binned, binning)
        suffix = f"_{GENOTYPE_SET}_{binning}"
        lmm.to_csv(stats_dir_5d / f"time_bins_mixedlm{suffix}.csv", index=False)
        tt.to_csv(stats_dir_5d / f"time_bins_across_genotypes{suffix}.csv", index=False)

# Plot 5f: Where Mice Turn Around

Per cell, how often a pass through it turns around there (a pass = one stay in the cell), per mouse then averaged: **backtrack** = of the passes going deeper (outbound, overshoot forward, reward zone), the share that turn back (forward -> backward) in the cell; **re-entry** = of the passes coming back (return, overshoot backward, reward zone), the share that turn in again (backward -> forward). Covers the path, the reward zone and the overshoot. The furthest point of a bout that never reaches the reward is its turnaround, not a backtrack. A mouse counts for a cell only with >= 5 such passes; grey = too few.


In [ ]:
from compass_labyrinth.post_hoc_analysis.level_1 import cell_turn_table, plot_reversal_heatmaps

# Per mouse x cell: passes and turns each way, from Plot 5's labeled_frames (raw genotypes)
turns_raw = cell_turn_table(labeled_frames)

for GENOTYPE_SET in COMPARISONS:
    plot_reversal_heatmaps(
        config=config,
        show_fig=GENOTYPE_SET == COMPARISONS[0],  # display the first comparison only; all are saved
        turns=build_genotype_groups(turns_raw, GENOTYPE_SET),
        grid_filename="Session0001_grid.shp",
        min_passes=5,
        filename_suffix=f"_{GENOTYPE_SET}",
        figures_dirname="figures/level_1/plot5f_turn_heatmap",
    )


# Plot 6a / 6b: Heatmaps by Direction of Travel

Plot 1/1b's maze heatmaps, one figure per direction of travel (Plot 5's phases grouped): **going in** (outbound + reward zone and overshoot moving deeper), **coming back** (return + reward zone and overshoot moving back), **backtrack**, **re-entry**, plus **coming back - going in** (each mouse's own difference, then averaged; red = higher coming back). 6a = Slow / Surveillance share, 6b = mean velocity. A mouse counts for a cell only with >= 20 frames there in that direction (both, for the difference); grey = too few frames -- backtrack and re-entry are sparse, so many cells are grey.

Also pooled by zone for `outbound_entry_exit` (with going in) and `return_entry_exit` (with coming back): the cells just before / after the U-turns, bottom and top zones (config.yaml).


In [ ]:
from compass_labyrinth.post_hoc_analysis.level_1 import (
    TRAVEL, cell_phase_table, with_travel, plot_direction_heatmaps,
    zone_direction_table, plot_zone_direction_heatmap, zone_direction_ttests,
)

# Zone-pooled versions, each zone set with the one direction it is defined for (the report
# greys it out for the other directions): cells just before / after the U-turns, bottom and top zones
ZONE_SET_DIRECTIONS = {"outbound_entry_exit": ["going_in"], "return_entry_exit": ["coming_back"]}

# Per mouse x cell x phase -- once, from Plot 5's labeled_frames (raw genotypes)
travel_cells_raw = cell_phase_table(with_travel(labeled_frames))  # phases grouped by travel direction

for GENOTYPE_SET in COMPARISONS:
    cells = build_genotype_groups(travel_cells_raw, GENOTYPE_SET)
    for metric, step in [("surveillance", "plot6a_direction_heatmap_surveillance"),
                         ("velocity", "plot6b_direction_heatmap_velocity")]:
        for direction in [*TRAVEL, "change"]:
            plot_direction_heatmaps(
                config=config,
                show_fig=(GENOTYPE_SET, direction) == (COMPARISONS[0], "going_in"),  # display only the first; all are saved
                cells=cells,
                metric=metric,
                direction=direction,
                grid_filename="Session0001_grid.shp",
                min_frames=20,
                filename_suffix=f"_{GENOTYPE_SET}",
                figures_dirname=f"figures/level_1/{step}",
            )

# Zone-set versions (6a/6b "Zone set" selector); stats per zone
stats_dirs = {step: Path(config["project_path_full"]) / "results" / "level_1" / step
              for step in ("plot6a_direction_heatmap_surveillance", "plot6b_direction_heatmap_velocity")}
for zone_set, directions in ZONE_SET_DIRECTIONS.items():
    zone_table_raw = zone_direction_table(travel_cells_raw, ZONES, ZONE_SETS[zone_set])
    for GENOTYPE_SET in COMPARISONS:
        zone_table = build_genotype_groups(zone_table_raw, GENOTYPE_SET)
        for (metric, step), direction in [(ms, d) for ms in [("surveillance", "plot6a_direction_heatmap_surveillance"),
                                                             ("velocity", "plot6b_direction_heatmap_velocity")]
                                          for d in directions]:
            suffix = f"_{zone_set}_{GENOTYPE_SET}"
            plot_zone_direction_heatmap(
                config=config,
                show_fig=False,
                table=zone_table,
                zones=ZONES,
                zone_names=ZONE_SETS[zone_set],
                metric=metric,
                direction=direction,
                filename_suffix=suffix,
                figures_dirname=f"figures/level_1/{step}",
            )
            stats_dirs[step].mkdir(parents=True, exist_ok=True)
            zone_direction_ttests(zone_table, metric, direction).to_csv(
                stats_dirs[step] / f"zone_ttests_{metric}_{direction}{suffix}.csv", index=False)


# Plot 6c / 6d: Surveillance and Velocity over Time by Direction

Plot 3 / 3b for the 8 entry / exit zone sets (`TEMPORAL_ZONE_SETS`: bottom / top x outbound / return x entry / exit, 3 zones each): same 2000-frame time bins and per-zone median over cells, but each cell's surveillance share (6c) or mean velocity (6d) counts only the frames moving in the zone set's direction -- outbound sets going in, return sets coming back.


In [ ]:
from compass_labyrinth.post_hoc_analysis.level_1 import (
    get_max_session_row_bracket,
    get_min_session_row_bracket,
    compute_node_state_medians_over_time,
    compute_node_velocity_medians_over_time,
    build_genotype_zone_palette,
    plot_node_state_median_curve,
    travel_direction,
)

lower_limit = 0
upper_limit = get_max_session_row_bracket(df_hmm)
threshold = get_min_session_row_bracket(df_hmm)  # Only show bins where all sessions are present
bin_size = 2000

# Direction of every analyzed frame (NaN between bouts), from Plot 5's labeled_frames
travel = df_hmm[["session", "s_no"]].merge(
    labeled_frames[["session", "s_no"]].assign(travel=travel_direction(labeled_frames)),
    on=["session", "s_no"], how="left")["travel"].to_numpy()

# Entry and exit zones of each side apart, each set with its own direction
TEMPORAL_ZONE_SETS = {f"{side}_{d}_{k}": ["going_in" if d == "outbound" else "coming_back"]
                      for side in ("bottom", "top") for d in ("outbound", "return") for k in ("entry", "exit")}

TEMPORAL = [  # (compute function, extra args, value column, label, y limits, filename stem, figure folder)
    (compute_node_state_medians_over_time, {"state_types": [1]}, "median_probability",
     "Median Probability of Surveillance", (0.3, 1.1), "temporal_median_curve", "plot6c_temporal_direction_surveillance"),
    (compute_node_velocity_medians_over_time, {}, "median_velocity",
     "Median Velocity", (0, 300), "temporal_median_velocity_curve", "plot6d_temporal_direction_velocity"),
]
for zone_set, directions in TEMPORAL_ZONE_SETS.items():
    for direction in directions:
        # Frames moving another way drop out of the per-cell counts; the time bins stay as in Plot 3
        df_dir = df_hmm.assign(grid_number=df_hmm["grid_number"].where(travel == direction))
        for compute, extra, value_col, label, ylim, stem, folder in TEMPORAL:
            medians_raw = compute(df_hmm=df_dir, lower_lim=lower_limit, upper_lim=upper_limit, bin_size=bin_size,
                                  zone_names=ZONE_SETS[zone_set], node_type_mapping=ZONES, **extra)
            medians_raw = medians_raw.loc[medians_raw.time_bins < threshold]
            for GENOTYPE_SET in COMPARISONS:
                medians = build_genotype_groups(medians_raw, GENOTYPE_SET)
                medians["genotype_node_type"] = medians["genotype"] + " , " + medians["node_type"]
                plot_node_state_median_curve(
                    config=config,
                    show_fig=False,
                    deci_df=medians,
                    palette=build_genotype_zone_palette(medians),
                    figure_ylimit=ylim,
                    value_col=value_col,
                    metric_label=label,
                    filename_stem=f"{stem}_{direction}",
                    fig_title=f"{label} across {zone_set} zones -- {TRAVEL[direction].split(' (')[0].lower()} frames only",
                    filename_suffix=f"_{GENOTYPE_SET}_{zone_set}",
                    figures_dirname=f"figures/level_1/{folder}",
                )


# Plot 7 / 7b: Head Angle by Direction of Travel

Which side the head points to relative to the body axis (tail base -> sternum; + = the mouse's right, - = its left), on the whole maze cell by cell, for each direction of travel as in 6a/6b -- **going in**, **coming back**, **backtrack**, **re-entry** -- one figure per direction x comparison. The phases are kept apart because the mouse's right points to opposite sides of the maze when it travels the other way; pooled, those leans would cancel. A mouse counts for a cell only with >= 20 frames there in that phase (grey = too few) -- backtrack and re-entry are sparse, so their cells rest on fewer mice. In each cell, the grey arrow is the body (tail base -> sternum) and the black arrow the head (sternum -> nose), turned by the mean head angle; the color and the number in the bottom-right corner give the angle. Assumes an unmirrored top-down camera. In the report, pick the phase with the **Direction** selector.

**7b** is the same map colored by the **absolute value** of Plot 7's signed mean (how strongly the head leans, ignoring the side); its arrows still show the side.

In [ ]:
from compass_labyrinth.post_hoc_analysis.level_1 import TRAVEL, cell_phase_angle_table, with_travel, plot_head_angle_cells

# Per mouse x cell x phase signed (and absolute) head angle -- Plot 5's labeled_frames + Plot 1c-1f's df_kin
angles_raw = cell_phase_angle_table(with_travel(labeled_frames), df_kin)  # phases grouped by travel direction

for GENOTYPE_SET in COMPARISONS:
    angles = build_genotype_groups(angles_raw, GENOTYPE_SET)
    for phase in TRAVEL:
        # 7: colored by the signed mean angle; 7b: by its absolute value (arrows show the signed angle in both)
        for absolute, folder in [(False, "plot7_head_angle_by_direction"), (True, "plot7b_head_angle_abs_by_direction")]:
            plot_head_angle_cells(
                config=config,
                show_fig=(GENOTYPE_SET, phase) == (COMPARISONS[0], "going_in"),  # display only the first; all are saved
                angles=angles,
                phase=phase,
                absolute=absolute,
                grid_filename="Session0001_grid.shp",
                min_frames=20,
                filename_suffix=f"_{GENOTYPE_SET}",
                figures_dirname=f"figures/level_1/{folder}",
            )

# Time-Binned Heatmaps

Plots 1, 1b-1f, 5f, 6a, 6b, 7 and 7b by time within the session: 5 equal parts (`pct_bins`) and frames 0-10k ... 40k-50k (`frame_bins`). Shown in the report under each step's **Time bins** selector.

In [ ]:
from compass_labyrinth.post_hoc_analysis.level_1 import plot_all_binned_heatmaps

plot_all_binned_heatmaps(
    config=config,
    df_hmm=df_hmm,
    kinematics=df_kin,
    labeled=labeled_frames,
    comparisons=COMPARISONS,
    genotype_sets=GENOTYPE_SETS,
    binnings=("pct_bins", "frame_bins"),
    min_frames=10,  # per mouse per cell and bin, for 6a, 6b, 7, 7b
    heatmap_vmin=HEATMAP_VMIN,
    zones=ZONES,
    zone_sets=ZONE_SETS,
    zone_set_directions=ZONE_SET_DIRECTIONS,  # 6a/6b by zone, from the 6a/6b cell
)

# Write the Data Sheet

Writes every table recorded by the plotting functions above (both raw and binned/aggregated -- whatever each figure was actually drawn from) into one Excel workbook, with mouse metadata joined on.

In [ ]:
source_data.keep("mice", source_data.metadata())
sheets = source_data.find("")
readme = {name: ("", "") for name in sheets}
source_data.write_workbook(
    str(Path(config["project_path_full"]) / "results" / "level_1" / f"{config['project_name']}_level_1_datasheet.xlsx"),
    sheets,
    readme,
)

# Rebuild the Figure Report

Regenerates `analysis_report.html` from whatever figures/stats are currently on disk for BOTH pipelines (task performance and level 1) -- run this notebook last if you want the report to reflect it.

In [ ]:
from compass_labyrinth.report import build_report

build_report(config["project_path_full"])